# PO4NCPA — performance visualization

Loads a trained PO4NCPA checkpoint and shows **how the policy behaves**, reusing the
exact pipeline from `src/po4ncpa.py` and `src/eval_po4ncpa.py` (so this stays in sync
with training/eval). Four views:

1. **Focal-plane image evolution** — the (dark) image at each step, dark-hole annulus overlaid.
2. **Contrast & Strehl vs step** — the refinement trajectory + the ideal-correction floor.
3. **Wavefront phase maps** — aberration, applied correction, residual.
4. **Held-out distribution** — final contrast/Strehl over N held-out draws vs the floor.

> **Run this on a compute node, not the Adroit login node** — every cell below builds
> `CoronagraphOptics` and propagates wavefronts. Either launch Jupyter inside an
> `salloc` session, or execute headless with the SLURM helper:
> `sbatch slurm/run_viz.slurm` (renders an executed copy + PNGs under `notebooks/figs/`).

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm

# import the project package (run from RL/ root or RL/notebooks/)
REPO = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
SRC = os.path.join(REPO, 'src')
if SRC not in sys.path:
    sys.path.insert(0, SRC)

from po4ncpa import Config, make_env, PolicyNet, calibrate_per_mode_scale
from eval_po4ncpa import preprocess, ideal_floor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('repo :', REPO)
print('device:', device)

In [ ]:
# ---- choose what to visualize -------------------------------------------------
CHECKPOINT = os.path.join(REPO, 'logs/po4ncpa_corona/po4ncpa_best.pt')  # or .../po4ncpa_strehl/...
DEMO_SEED  = 900000      # held-out seed for the single-episode walkthrough (views 1-3)
N_HELDOUT  = 60          # episodes for the distribution (view 4); 150 in the SLURM eval
SEED_OFFSET = 900000     # held-out seed block (disjoint from training)
FIGDIR = os.path.join(REPO, 'notebooks/figs'); os.makedirs(FIGDIR, exist_ok=True)

In [ ]:
# ---- load checkpoint, rebuild env + per-mode scaling + policy ------------------
ckpt = torch.load(CHECKPOINT, map_location=device, weights_only=False)
cfg = Config(**ckpt['cfg'])
env = make_env(cfg)
if cfg.per_mode_scale:
    scale = calibrate_per_mode_scale(env.optics, cfg)
    env.action_scale = scale
    env.max_abs_actuator = scale
h, w = env.image_shape
action_dim = int(env.action_space.shape[0])
ideal = env.ideal_image().astype(np.float32)        # PSF_ideal reference for preprocessing
max_steps = env.max_steps

policy = PolicyNet(h, w, action_dim, ch=cfg.ch).to(device)
policy.load_state_dict(ckpt['policy'])
policy.eval()

corona = env.use_coronagraph
print(f'run={cfg.run_name}  coronagraph={corona}  modes={action_dim}  '
      f'image={h}x{w}  steps={max_steps}  start_mode={cfg.aberration_start_mode}  '
      f'rms={cfg.rms_min}-{cfg.rms_max}')

In [ ]:
# ---- deterministic rollout that ALSO captures images and final phase maps -----
def rollout(seed):
    obs, info = env.reset(seed=seed)
    imgs    = [obs['image'][0].copy()]          # raw normalized-intensity science/dark image
    contr   = [info['contrast']]
    strehl  = [info['strehl']]
    o_cur   = preprocess(obs['image'][0], ideal)
    o_prev  = o_cur.copy()
    a_prev  = obs['command'].astype(np.float32).copy()
    for _ in range(max_steps):
        pair = torch.as_tensor(np.stack([o_cur, o_prev])[None], device=device)
        ap   = torch.as_tensor(a_prev[None], device=device)
        with torch.no_grad():
            a = policy(pair, ap)[0].cpu().numpy().astype(np.float32)
        obs, _r, _t, _tr, info = env.step(a)
        imgs.append(obs['image'][0].copy())
        contr.append(info['contrast']); strehl.append(info['strehl'])
        o_prev, o_cur = o_cur, preprocess(obs['image'][0], ideal)
        a_prev = obs['command'].astype(np.float32).copy()
    # final wavefront phases (pupil grid), masked to the aperture for display
    opt = env.optics
    sup = opt._aperture_support
    side = int(np.sqrt(opt.aberration_phase.size))
    def pup(ph):
        ph = np.asarray(ph).astype(float).copy(); ph[~sup] = np.nan
        return ph.reshape(side, side)
    phases = dict(aberration=pup(opt.aberration_phase),
                  correction=pup(opt.correction_phase),
                  residual=pup(np.asarray(opt.aberration_phase) + np.asarray(opt.correction_phase)))
    return (np.array(imgs), np.array(contr), np.array(strehl), phases)

imgs, contr, strehl, phases = rollout(DEMO_SEED)
print('captured', imgs.shape[0], 'frames; final contrast %.3e  strehl %.4f' % (contr[-1], strehl[-1]))

## View 1 — focal-plane image evolution
The dark-hole annulus (scoring region) is outlined. Watch the speckles inside it drain
as the policy refines over the first few steps.

In [ ]:
dh_mask = np.asarray(env.optics.dark_hole_mask).reshape(h, w)
show_steps = sorted(set([0, 1, 2, 3, 4, max_steps]))
vmin = max(np.min([im[im > 0].min() for im in imgs]), 1e-12)
vmax = float(np.max(imgs))
fig, axes = plt.subplots(1, len(show_steps), figsize=(3.0 * len(show_steps), 3.4))
for ax, t in zip(axes, show_steps):
    im = np.clip(imgs[t], vmin, None)
    h_im = ax.imshow(im, norm=LogNorm(vmin=vmin, vmax=vmax), cmap='inferno', origin='lower')
    ax.contour(dh_mask.astype(float), levels=[0.5], colors='cyan', linewidths=0.8)
    ax.set_title(f'step {t}\nC={contr[t]:.2e}  S={strehl[t]:.3f}', fontsize=9)
    ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(h_im, ax=axes, shrink=0.7, label='normalized intensity (log)')
fig.suptitle(f'Focal-plane image — {cfg.run_name} (coronagraph={corona})', y=1.02)
fig.savefig(os.path.join(FIGDIR, 'view1_image_evolution.png'), dpi=130, bbox_inches='tight')
plt.show()

## View 2 — contrast & Strehl trajectory vs the ideal floor
`ideal floor` = exact 55-mode projection of the true aberration (the best this geometry
can do). The gap between the policy curve and the floor is the **sensing limit**.

In [ ]:
env.reset(seed=DEMO_SEED)
floor_c, floor_s = ideal_floor(env)
steps = np.arange(len(contr))
fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.semilogy(steps, contr, 'o-', color='tab:red', label='policy')
a1.axhline(floor_c, ls='--', color='k', label=f'ideal floor {floor_c:.2e}')
a1.set_xlabel('step'); a1.set_ylabel('dark-hole contrast'); a1.set_title('Contrast'); a1.legend(); a1.grid(alpha=0.3)
a2.plot(steps, strehl, 'o-', color='tab:blue', label='policy')
a2.axhline(floor_s, ls='--', color='k', label=f'ideal floor {floor_s:.4f}')
a2.set_xlabel('step'); a2.set_ylabel('Strehl'); a2.set_title('Strehl'); a2.legend(); a2.grid(alpha=0.3)
fig.suptitle(f'Refinement trajectory — seed {DEMO_SEED}')
fig.savefig(os.path.join(FIGDIR, 'view2_trajectory.png'), dpi=130, bbox_inches='tight')
plt.show()
print(f'policy final contrast {contr[-1]:.3e}  |  ideal floor {floor_c:.3e}  |  ratio {contr[-1]/max(floor_c,1e-30):.1f}x')

## View 3 — wavefront phase maps
Aberration that was thrown at the system, the correction the policy applied, and the
**residual** (what's left). A flat residual = perfect; the residual RMS is what scatters
the speckle floor.

In [ ]:
vlim = np.nanmax(np.abs(phases['aberration']))
fig, axes = plt.subplots(1, 3, figsize=(12, 4))
for ax, key in zip(axes, ['aberration', 'correction', 'residual']):
    m = ax.imshow(phases[key], cmap='RdBu_r', vmin=-vlim, vmax=vlim, origin='lower')
    rms = np.sqrt(np.nanmean(phases[key] ** 2))
    ax.set_title(f'{key}\nRMS = {rms:.3f} rad', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])
    fig.colorbar(m, ax=ax, shrink=0.75)
fig.suptitle('Pupil-plane phase (radians)')
fig.savefig(os.path.join(FIGDIR, 'view3_phase_maps.png'), dpi=130, bbox_inches='tight')
plt.show()

## View 4 — held-out distribution
Final contrast & Strehl over N held-out draws (deterministic), with the per-draw ideal
floor for reference. This is the population version of the single episode above.

In [ ]:
fin_c, fin_s, flo_c = [], [], []
for e in range(N_HELDOUT):
    seed = SEED_OFFSET + e
    env.reset(seed=seed); fc, _ = ideal_floor(env)
    _, c, s, _ = rollout(seed)
    fin_c.append(c[-1]); fin_s.append(s[-1]); flo_c.append(fc)
fin_c, fin_s, flo_c = map(np.array, (fin_c, fin_s, flo_c))

fig, (a1, a2) = plt.subplots(1, 2, figsize=(11, 4))
a1.hist(np.log10(fin_c), bins=20, color='tab:red', alpha=0.8)
a1.axvline(np.log10(np.median(fin_c)), color='k', label=f'median {np.median(fin_c):.2e}')
a1.axvline(np.log10(np.median(flo_c)), color='k', ls='--', label=f'ideal floor {np.median(flo_c):.2e}')
a1.set_xlabel('log10(final contrast)'); a1.set_ylabel('count'); a1.set_title('Final contrast'); a1.legend(fontsize=8)
a2.hist(fin_s, bins=20, color='tab:blue', alpha=0.8)
a2.axvline(np.median(fin_s), color='k', label=f'median {np.median(fin_s):.4f}')
a2.set_xlabel('final Strehl'); a2.set_ylabel('count'); a2.set_title(f'Final Strehl ({np.mean(fin_s>0.99)*100:.0f}% > 0.99)'); a2.legend(fontsize=8)
fig.suptitle(f'{N_HELDOUT} held-out episodes — {cfg.run_name}')
fig.savefig(os.path.join(FIGDIR, 'view4_distribution.png'), dpi=130, bbox_inches='tight')
plt.show()
print(f'contrast: median {np.median(fin_c):.3e}  best {fin_c.min():.3e}  worst {fin_c.max():.3e}')
print(f'strehl  : median {np.median(fin_s):.4f}  | ideal floor median {np.median(flo_c):.3e}')

### How to read these
- **View 1**: speckles inside the cyan annulus should fade over steps 1–4, then hold.
- **View 2**: the persistent gap between the red curve and the dashed floor is the
  information/sensing limit — the policy can't sense what it can't see in the dark image.
- **View 3**: a near-flat residual with small RMS is the goal; leftover structure shows
  which spatial frequencies the policy fails to correct.
- **View 4**: the spread tells you robustness across aberration draws; the floor line
  shows how far from physically-optimal the controller sits.